In [1]:
from datasets import load_dataset

dataset = load_dataset('Blpeng/nsmc')
for i in range(3):
    print(dataset['train'][i]['document'], dataset['train'][i]['label'])

Repo card metadata block was not found. Setting CardData to empty.


아 더빙.. 진짜 짜증나네요 목소리 0
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나 1
너무재밓었다그래서보는것을추천한다 0


In [2]:
from transformers import AutoTokenizer
MODEL_NAME = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_col = 'document'
format_columns = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
def tokenize_fn(batch):
    """배치 단위로 텍스트 토크나이즈 (최대 128 토큰)"""
    # None/비문자열 샘플이 섞여 있어도 안전하게 문자열로 변환
    texts = [x if isinstance(x, str) else '' for x in batch[text_col]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding='max_length'
    )
# 전체 데이터셋에 적용 (batched=True로 빠르게)
tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')  # Trainer가 'labels' 필드 기대
tokenized.set_format('torch', columns=format_columns)

In [4]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f'모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'분류 헤드: {model.classifier}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 파라미터 수: 110,618,882
분류 헤드: Linear(in_features=768, out_features=2, bias=True)
